In [2]:
import pandas as pd
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

def setup_dummy_data():
    os.makedirs('ecommerce_data', exist_ok=True)
    
    if os.path.exists('ecommerce_data/orders.csv'):
        print("  -> Data files already exist. Skipping dummy generation so your new data is safe.")
        return

    with open('ecommerce_data/customers.csv', 'w') as f:
        f.write("customer_id,name,email,signup_date\n")
        f.write("1,alice smith,alice@email.com,2023-01-10\n")
        f.write("2,bob jones,,2023-01-15\n") 
        f.write("3,charlie brown,charlie@email.com,2023-02-01\n")

    with open('ecommerce_data/products.csv', 'w') as f:
        f.write("product_id,product_name,category,price\n")
        f.write("101,Laptop,Electronics,1200.00\n")
        f.write("102,Wireless Mouse,Electronics,25.50\n")
        f.write("103,Desk Chair,Furniture,-50.00\n") 

    with open('ecommerce_data/orders.csv', 'w') as f:
        f.write("order_id,customer_id,product_id,quantity,order_date\n")
        f.write("1001,1,101,1,2023-10-01\n") 
        f.write("1002,2,102,2,2023-10-02\n") 
        f.write("1003,3,101,1,2023-10-05\n") 
        f.write("1004,1,102,1,2023-10-06\n")

def load_incrementally(df, table_name, primary_key, engine):
    try:
        query = f"SELECT {primary_key} FROM {table_name}"
        existing_keys = pd.read_sql(query, engine)[primary_key].tolist()
        
        df_new = df[~df[primary_key].isin(existing_keys)]
    except Exception:
        df_new = df

    if not df_new.empty:
        df_new.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"  -> SUCCESS: Inserted {len(df_new)} new records into '{table_name}'.")
    else:
        print(f"  -> SUCCESS: '{table_name}' is already up to date. No new records inserted.")

def run_pipeline():
    print("--- Starting Practical 9 E-Commerce ETL Pipeline ---\n")
    
    print("[0/5] Checking for data files...")
    setup_dummy_data() 
    
    print("\n[1/5] EXTRACT: Reading Data from CSVs...")
    print("\n[1/5] EXTRACT: Reading Data from CSVs...")
    df_cust = pd.read_csv('ecommerce_data/customers.csv')
    df_prod = pd.read_csv('ecommerce_data/products.csv')
    df_ord = pd.read_csv('ecommerce_data/orders.csv')

    print("\n[2/5] TRANSFORM: Cleaning and Formatting Data...")
    df_cust['name'] = df_cust['name'].str.title()
    df_cust['email'] = df_cust['email'].fillna('Unknown')
    df_cust['signup_date'] = pd.to_datetime(df_cust['signup_date'])
    
    df_prod = df_prod[df_prod['price'] > 0]
    
    df_ord['order_date'] = pd.to_datetime(df_ord['order_date'])

    print("\n[3/5] LOAD: Connecting to MS SQL Server...")
    server_name = r'DESKTOP-P5RT5RO'  
    database_name = 'ECommerceDB'
    
    connection_url = URL.create(
        "mssql+pyodbc",
        host=server_name,
        database=database_name,
        query={"driver": "ODBC Driver 17 for SQL Server", "Trusted_Connection": "yes"}
    )
    engine = create_engine(connection_url, fast_executemany=True)

    print("\n[4/5] LOAD: Executing Incremental Database Inserts...")
    load_incrementally(df_cust, 'dim_customers', 'customer_id', engine)
    load_incrementally(df_prod, 'dim_products', 'product_id', engine)
    load_incrementally(df_ord, 'fact_orders', 'order_id', engine)

    print("\n[5/5] REPORT: Generating Automated Analytics Report...")
    report_query = """
        SELECT 
            p.product_name,
            SUM(o.quantity) as total_units_sold,
            SUM(o.quantity * p.price) as total_revenue
        FROM fact_orders o
        JOIN dim_products p ON o.product_id = p.product_id
        GROUP BY p.product_name
        ORDER BY total_revenue DESC;
    """
    
    df_report = pd.read_sql(report_query, engine)
    
    print("\n================ BUSINESS REPORT ================")
    print(df_report.to_string(index=False))
    print("=================================================\n")
    
    df_report.to_csv('ecommerce_data/final_sales_report.csv', index=False)
    print("Pipeline Complete! Report saved to 'ecommerce_data/final_sales_report.csv'.")

if __name__ == "__main__":
    run_pipeline()

--- Starting Practical 9 E-Commerce ETL Pipeline ---

[0/5] Checking for data files...

[1/5] EXTRACT: Reading Data from CSVs...

[1/5] EXTRACT: Reading Data from CSVs...

[2/5] TRANSFORM: Cleaning and Formatting Data...

[3/5] LOAD: Connecting to MS SQL Server...

[4/5] LOAD: Executing Incremental Database Inserts...


c:\Users\admin\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


  -> SUCCESS: Inserted 3 new records into 'dim_customers'.
  -> SUCCESS: Inserted 2 new records into 'dim_products'.
  -> SUCCESS: Inserted 4 new records into 'fact_orders'.

[5/5] REPORT: Generating Automated Analytics Report...

================ BUSINESS REPORT ================
  product_name  total_units_sold  total_revenue
        Laptop                 2         2400.0
Wireless Mouse                 3           76.5

Pipeline Complete! Report saved to 'ecommerce_data/final_sales_report.csv'.
